In [13]:
import os
import tarfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Subset, Dataset
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
data_dir = "~/work/data_augmentation/data/Images/"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
BATCH_SIZE = 32
EPOCHS = 20
NUM_CLASSES = 120
IMAGE_SIZE = 224
SEED = 42

In [5]:
def augment(image, label):
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2),
    ])
    return transform(image), label

In [6]:
def normalize_and_resize_img(image, label):
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225])
    ])
    return transform(image), label

In [7]:
# ----------------------------
# 원-핫 인코딩 (MixUp/CutMix용 soft label 만들 때 필요)
# ----------------------------
def onehot(label, num_classes=NUM_CLASSES):
    # label: (B,) tensor(long) 또는 int
    if isinstance(label, int):
        label = torch.tensor(label, dtype=torch.long)
    return torch.nn.functional.one_hot(label, num_classes=num_classes).float()

In [8]:
def soft_cross_entropy_from_logits(logits, targets_soft):
    log_probs = torch.log_softmax(logits, dim=1)
    return -(targets_soft * log_probs).sum(dim=1).mean()


In [9]:
def apply_mixup(x, y_onehot, alpha=1.0):
    if alpha <= 0:
        return x, y_onehot

    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_mix = lam * y_onehot + (1 - lam) * y_onehot[idx]
    return x_mix, y_mix

In [10]:
def rand_bbox(H, W, lam):
    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_rat)
    cut_w = int(W * cut_rat)

    cy = np.random.randint(H)
    cx = np.random.randint(W)

    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2

In [11]:
def apply_cutmix(x, y_onehot, alpha=1.0):
    if alpha <= 0:
        return x, y_onehot

    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)

    B, C, H, W = x.size()
    y1, y2, x1, x2 = rand_bbox(H, W, lam)

    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]

    area = (y2 - y1) * (x2 - x1)
    lam_adj = 1.0 - area / float(H * W)

    y_cut = lam_adj * y_onehot + (1.0 - lam_adj) * y_onehot[idx]
    return x_cut, y_cut


In [14]:
class TransformWrapper(Dataset):
    def __init__(self, base_subset, is_test=False, with_aug=False):
        self.base = base_subset
        self.is_test = is_test
        self.with_aug = with_aug

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, lbl = self.base[idx]  # PIL, int
        if (not self.is_test) and self.with_aug:
            img, lbl = augment(img, lbl)
        img, lbl = normalize_and_resize_img(img, lbl)
        return img, lbl

def apply_normalize_on_dataset(dataset, is_test=False, batch_size=32, with_aug=False):
    wrapped = TransformWrapper(dataset, is_test=is_test, with_aug=with_aug)

    loader = DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=not is_test,
        num_workers=0,      # 중요: 2 -> 0
        pin_memory=False    # 중요: True -> False
    )
    return loader

In [15]:
def make_split_indices(n, seed=SEED):
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    train_size = int(0.8 * n)
    train_idx = idx[:train_size].tolist()
    val_idx = idx[train_size:].tolist()
    return train_idx, val_idx


In [16]:
def train_experiment(mode='none', mixup_alpha=1.0, cutmix_alpha=1.0):
    # base dataset (PIL + label)
    base_full = torchvision.datasets.ImageFolder(root=data_dir)  # transform=None => PIL 반환
    n = len(base_full)
    train_idx, val_idx = make_split_indices(n)

    train_ds = Subset(base_full, train_idx)
    val_ds   = Subset(base_full, val_idx)

    # mode에 따라 "기본 augmentation" 유무만 loader 단계에서 결정
    with_aug = (mode != 'none')  # basic/mixup/cutmix는 기본 증강 포함
    train_loader = apply_normalize_on_dataset(train_ds, is_test=False, batch_size=BATCH_SIZE, with_aug=with_aug)
    val_loader   = apply_normalize_on_dataset(val_ds, is_test=True,  batch_size=BATCH_SIZE, with_aug=False)

    # 모델
    model = models.resnet50(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    model = model.to(device)

    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    history = {'loss': [], 'acc': []}

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).long()

            y_onehot = onehot(y, NUM_CLASSES).to(device)

            # mode에 따라 배치 섞기
            if mode == 'mixup':
                x, y_onehot = apply_mixup(x, y_onehot, alpha=mixup_alpha)
            elif mode == 'cutmix':
                x, y_onehot = apply_cutmix(x, y_onehot, alpha=cutmix_alpha)
            # 'none'/'basic'는 섞기 없음

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = soft_cross_entropy_from_logits(logits, y_onehot)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # validation (hard accuracy)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True).long()
                logits = model(x)
                pred = logits.argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        acc = 100.0 * correct / total
        history['loss'].append(total_loss / len(train_loader))
        history['acc'].append(acc)
        print(f"[{mode}] Epoch {epoch+1}/{EPOCHS}: Loss {history['loss'][-1]:.4f}, Acc {acc:.2f}%")

    return history


In [ ]:
modes = ['none', 'basic', 'mixup', 'cutmix']
results = {}

for m in modes:
    results[m] = train_experiment(m)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/jovyan/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


[none] Epoch 1/20: Loss 2.3731, Acc 78.84%
